# 🌍 Week 4 — Geospatial Foundations
### Geospatial Python Mastery | Module B: Data Structures and File I/O

---

|  |  |
|---|---|
| **Course** | Geospatial Python Mastery |
| **Week** | 4 of 10 |
| **Theme** | Geospatial Foundations — Coordinates, Projections, CRS |
| **Duration** | ~4 contact hours + 4 hours self-study |
| **Practice Outcome** | Mini-Lab: Compare WGS84 and UTM 31N workflows for Netherlands cities |
| **Next week** | Spatial libraries — GeoPandas, Shapely full API, Rasterio, Fiona |

---

> 🗺️ **Why this week matters**
> Coordinates only become meaningful when you know the datum, the axis order, and the
> units behind them. Week 4 connects geography and code: you will inspect CRS metadata,
> transform Dutch city coordinates into projected systems, compare distance methods, and
> use Shapely geometry objects to reason about topology before moving into full GeoPandas workflows.

---

## 📋 Table of Contents

| Section | Topic |
|---------|-------|
| [1 — Environment Setup](#section-1) | Auto-install, imports, data dirs |
| [2 — Coordinate Systems Basics](#section-2) | WGS84, lat/lon, geographic vs projected |
| [3 — PyProj CRS](#section-3) | EPSG inspection, axis order, units |
| [4 — Coordinate Transformations](#section-4) | `Transformer.from_crs(always_xy=True)` |
| [5 — Distance Comparison](#section-5) | Haversine, `Geod.inv`, UTM Euclidean |
| [6 — Shapely Geometry Types](#section-6) | Point, LineString, Polygon properties |
| [7 — Topology and Spatial Predicates](#section-7) | `contains`, `within`, `touches`, DE-9IM |
| [8 — Reusable Module](#section-8) | Write and import `crs_utils.py` |
| [9 — File Formats](#section-9) | Save and reload WKT, GeoJSON, CSV |
| [10 — Logging and Testing](#section-10) | Log transforms and test helper functions |

---

## 🗺️ Symbol Guide

| Symbol | Meaning |
|--------|---------|
| 💻 | Runnable code cell |
| 🎯 | Exercise — write your own code |
| ✅ | Solution — run after attempting |
| 🔬 | Mini-Lab step |
| 📖 | Explanatory section |
| 💡 | Tip or best practice |
| ⚠️ | Common mistake or warning |

> **Keyboard shortcuts:** `Shift+Enter` run cell · `b` insert cell below · `m` convert to Markdown · `Esc` command mode

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Objective |
|---|-----------|
| 1 | Explain latitude, longitude, and altitude in the WGS84 datum |
| 2 | Distinguish **geographic** (degree-based) from **projected** (metre-based) CRS |
| 3 | Use `pyproj.CRS` to inspect EPSG codes, axis order, and units |
| 4 | Transform coordinates with `pyproj.Transformer.from_crs(always_xy=True)` |
| 5 | Create Shapely **Point**, **LineString**, and **Polygon** objects and read their properties |
| 6 | Test spatial relationships: `contains`, `intersects`, `within`, `touches` |
| 7 | Build a reusable `crs_utils.py` module with validation and transform helpers |
| 8 | Export geometry data to WKT, GeoJSON, and CSV and re-read it portably |

### How to use this notebook

- Run cells **top to bottom** in sequence
- **Attempt** each 🎯 exercise before looking at the ✅ solution
- The 🔬 **Mini-Lab** at the end ties all skills together
- Tick each box in the ☑️ checklist before moving to the next week

In [ ]:
# 💻 Environment check and auto-install
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, importlib, platform

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OS_NAME = platform.system()

print("=" * 60)
print("  Geospatial Python Mastery — Week 4 Environment Check")
print("=" * 60)
print(f"  Environment : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"  Python      : {sys.version.split()[0]}")
print(f"  OS          : {OS_NAME}")

REQUIRED = [
    ("math",     "",        ""),
    ("json",     "",        ""),
    ("csv",      "",        ""),
    ("logging",  "",        ""),
    ("unittest", "",        ""),
    ("pyproj",   "pyproj",  "3.0"),
    ("shapely",  "shapely", "2.0"),
]

for import_name, pip_name, min_ver in REQUIRED:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "stdlib")
        print(f"  ✅ {import_name:<20} {ver}")
    except ImportError:
        pkg = pip_name or import_name
        print(f"  ⬇  Installing {pkg}…")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"  ✅ {pkg} installed")

if OS_NAME == "Windows":
    print("\n  ℹ  Windows note: use 'python -m jupyter notebook' to launch Jupyter")

print("\n✅ Environment check complete — ready for Week 4!")

In [ ]:
# 💻 0.1  Import CRS, geometry, and file helpers
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import logging
import math
import sys
import unittest
from importlib import reload
from pathlib import Path
from pprint import pprint

from pyproj import CRS, Geod, Transformer
from shapely import from_wkt, to_wkt
from shapely.geometry import LineString, Point, Polygon, mapping, shape

print('Imports loaded successfully.')

In [ ]:
# 💻 0.2  Create data, logs, modules, and test folders
# ─────────────────────────────────────────────────────────────────────────────
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'instructions').exists():
            return candidate
    return start

PROJECT_ROOT = find_repo_root(Path.cwd())
COURSE_DIR = PROJECT_ROOT / 'notebooks' / 'geospatial_python_course'
WEEK_DIR = COURSE_DIR / 'data' / 'week_04'
INPUT_DIR = WEEK_DIR / 'input'
OUTPUT_DIR = WEEK_DIR / 'output'
LOG_DIR = COURSE_DIR / 'logs'
MODULE_DIR = COURSE_DIR / 'local_modules'
TEST_DIR = COURSE_DIR / 'tests'

for path in [WEEK_DIR, INPUT_DIR, OUTPUT_DIR, LOG_DIR, MODULE_DIR, TEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Week 4 input :', INPUT_DIR)
print('Week 4 output:', OUTPUT_DIR)

---
## 📖 Section 1 — Environment Setup

Create a predictable workspace before you start transforming coordinates or writing geometry files.

### 1.1 Course folders and starter city data
We keep raw inputs, exported outputs, test files, and local utility modules in stable locations so later spatial workflows stay portable.

In [ ]:
# 💻 1.1  Inspect shared folders and CRS targets
# ─────────────────────────────────────────────────────────────────────────────
workspace = {
    'input': INPUT_DIR,
    'output': OUTPUT_DIR,
    'logs': LOG_DIR,
    'modules': MODULE_DIR,
    'tests': TEST_DIR,
}
for label, path in workspace.items():
    print(f'{label:<8} -> {path}')

WGS84 = 4326
UTM31N = 32631
RD_NEW = 28992
print('\nTarget EPSG codes:', WGS84, UTM31N, RD_NEW)

In [ ]:
# 💻 1.2  Create a starter CSV of Dutch cities in WGS84
# ─────────────────────────────────────────────────────────────────────────────
dutch_cities = [
    {'city': 'Amsterdam', 'lon': 4.9041, 'lat': 52.3676, 'country': 'NL'},
    {'city': 'Rotterdam', 'lon': 4.4777, 'lat': 51.9244, 'country': 'NL'},
    {'city': 'Utrecht', 'lon': 5.1214, 'lat': 52.0907, 'country': 'NL'},
    {'city': 'Groningen', 'lon': 6.5665, 'lat': 53.2194, 'country': 'NL'},
    {'city': 'Eindhoven', 'lon': 5.4697, 'lat': 51.4416, 'country': 'NL'},
]

city_csv_path = INPUT_DIR / 'netherlands_cities_wgs84.csv'
with city_csv_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['city', 'country', 'lon', 'lat'])
    writer.writeheader()
    writer.writerows(dutch_cities)

print('Saved starter CSV to:', city_csv_path)
pprint(dutch_cities)

### 🎯 Exercise 1 — Create a CRS manifest file

**Task:** Use `pathlib` and `json` to write a manifest describing the EPSG codes and data files used this week.

**Steps:**
1. Create a Python dictionary with keys like `source_crs`, `projected_targets`, and `input_file`.
2. Write the manifest to `INPUT_DIR / "week4_manifest.json"`.

```python
# Hint
manifest = {"source_crs": 4326, "projected_targets": [32631, 28992]}
```

In [ ]:
# 🎯 Exercise 1 — your code here ────────────────────────────────────────────
# 1. Create the manifest dictionary.
# 2. Save it to JSON.

In [ ]:
# ✅ Exercise 1 — Solution ────────────────────────────────────────────────────
manifest = {
    'source_crs': WGS84,
    'projected_targets': [UTM31N, RD_NEW],
    'input_file': city_csv_path.name,
}
manifest_path = INPUT_DIR / 'week4_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Manifest saved to:', manifest_path)
print(manifest_path.read_text(encoding='utf-8'))

---
## 📖 Section 2 — Coordinate Systems Basics

Coordinates are only numbers until you attach meaning: datum, axis order, units, and area of use.

### 2.1 Geographic vs projected CRS
A geographic CRS stores angular coordinates on an ellipsoid, while a projected CRS converts those angles into planar x/y values measured in metres or feet.

In [ ]:
# 💻 2.1  Review WGS84 latitude and longitude values
# ─────────────────────────────────────────────────────────────────────────────
print('City              lon (°)    lat (°)')
print('-' * 38)
for row in dutch_cities:
    print(f"{row['city']:<12} {row['lon']:>7.4f}   {row['lat']:>7.4f}")

print('\nWGS84 is geographic: units are degrees, not metres.')
print('Altitude can be added as a third value, but most vector workflows begin with lon/lat.')

In [ ]:
# 💻 2.2  Compare angular and metric thinking
# ─────────────────────────────────────────────────────────────────────────────
print("""Geographic CRS (WGS84)
  x = longitude in degrees
  y = latitude in degrees
Projected CRS (UTM / RD New)
  x = easting in metres
  y = northing in metres""")

amsterdam = dutch_cities[0]
rotterdam = dutch_cities[1]
lon_delta = abs(amsterdam['lon'] - rotterdam['lon'])
lat_delta = abs(amsterdam['lat'] - rotterdam['lat'])
print(f'\nAmsterdam ↔ Rotterdam degree delta: lon={lon_delta:.4f}, lat={lat_delta:.4f}')
print('Those degree differences are useful for storage, but not a direct metric distance.')

### 🎯 Exercise 2 — Describe coordinate meaning

**Task:** Pick two cities from `dutch_cities`, compute the raw degree deltas, and explain in a comment why that is not the same as a true ground distance.

**Steps:**
1. Subtract longitude and latitude values for any city pair.
2. Add a short code comment explaining why projection or geodesic distance is needed.

```python
# Hint
d_lon = city_a["lon"] - city_b["lon"]
```

In [ ]:
# 🎯 Exercise 2 — your code here ────────────────────────────────────────────
# 1. Compute longitude and latitude deltas.
# 2. Write a comment about degrees versus metres.

In [ ]:
# ✅ Exercise 2 — Solution ────────────────────────────────────────────────────
city_a = dutch_cities[2]
city_b = dutch_cities[4]
d_lon = abs(city_a['lon'] - city_b['lon'])
d_lat = abs(city_a['lat'] - city_b['lat'])

# Degrees are angular measurements on the ellipsoid, so they do not give a single metric distance.
print(f"{city_a['city']} ↔ {city_b['city']} degree delta: lon={d_lon:.4f}, lat={d_lat:.4f}")

---
## 📖 Section 3 — PyProj CRS

`pyproj.CRS` turns an EPSG code into structured metadata you can inspect directly in Python.

### 3.1 Names, axes, and units
CRS inspection is how you confirm whether a coordinate system is geographic or projected before you compute lengths or areas.

In [ ]:
# 💻 3.1  Inspect WGS84 and UTM 31N objects
# ─────────────────────────────────────────────────────────────────────────────
crs_wgs84 = CRS.from_epsg(WGS84)
crs_utm31n = CRS.from_epsg(UTM31N)

print('WGS84 name       :', crs_wgs84.name)
print('WGS84 is geographic:', crs_wgs84.is_geographic)
print('WGS84 axis info  :', [(axis.name, axis.unit_name) for axis in crs_wgs84.axis_info])
print('\nUTM31N name      :', crs_utm31n.name)
print('UTM31N projected :', crs_utm31n.is_projected)
print('UTM31N axis info :', [(axis.name, axis.unit_name) for axis in crs_utm31n.axis_info])

In [ ]:
# 💻 3.2  Inspect RD New and export WKT
# ─────────────────────────────────────────────────────────────────────────────
crs_rd_new = CRS.from_epsg(RD_NEW)
print('RD New name      :', crs_rd_new.name)
print('RD New axis info :', [(axis.name, axis.unit_name) for axis in crs_rd_new.axis_info])
print('Equals EPSG string:', crs_utm31n.equals(CRS.from_user_input('EPSG:32631')))
print('\nFirst 220 characters of WKT:')
print(crs_rd_new.to_wkt()[:220] + '...')

### 🎯 Exercise 3 — Inspect an EPSG code

**Task:** Create a `CRS` object for EPSG:28992 and print its name, whether it is projected, and its unit.

**Steps:**
1. Use `CRS.from_epsg(28992)`.
2. Read metadata from `.name`, `.is_projected`, and `.axis_info`.

```python
# Hint
rd = CRS.from_epsg(28992)
```

In [ ]:
# 🎯 Exercise 3 — your code here ────────────────────────────────────────────
# 1. Create the CRS object.
# 2. Print the key metadata fields.

In [ ]:
# ✅ Exercise 3 — Solution ────────────────────────────────────────────────────
rd = CRS.from_epsg(28992)
print('Name        :', rd.name)
print('Projected   :', rd.is_projected)
print('Unit        :', rd.axis_info[0].unit_name)

---
## 📖 Section 4 — Coordinate Transformations

Transformations convert the same location into a coordinate system that matches the measurement task you want to perform.

### 4.1 Using Transformer with always_xy=True
Set `always_xy=True` so you consistently provide longitude first and latitude second, even for CRS definitions that advertise latitude-first axis order.

In [ ]:
# 💻 4.1  Transform Amsterdam from WGS84 to UTM 31N
# ─────────────────────────────────────────────────────────────────────────────
transformer_utm = Transformer.from_crs(WGS84, UTM31N, always_xy=True)
transformer_rd = Transformer.from_crs(WGS84, RD_NEW, always_xy=True)

ams = dutch_cities[0]
ams_x_utm, ams_y_utm = transformer_utm.transform(ams['lon'], ams['lat'])
ams_x_rd, ams_y_rd = transformer_rd.transform(ams['lon'], ams['lat'])

print('Amsterdam WGS84 :', (ams['lon'], ams['lat']))
print('Amsterdam UTM31N:', (round(ams_x_utm, 2), round(ams_y_utm, 2)))
print('Amsterdam RD New:', (round(ams_x_rd, 2), round(ams_y_rd, 2)))

In [ ]:
# 💻 4.2  Batch-transform all Dutch cities
# ─────────────────────────────────────────────────────────────────────────────
transformed_cities = []
for row in dutch_cities:
    x_utm, y_utm = transformer_utm.transform(row['lon'], row['lat'])
    x_rd, y_rd = transformer_rd.transform(row['lon'], row['lat'])
    transformed_cities.append({
        **row,
        'utm31n_x': round(x_utm, 2),
        'utm31n_y': round(y_utm, 2),
        'rd_x': round(x_rd, 2),
        'rd_y': round(y_rd, 2),
    })

pprint(transformed_cities)

### 🎯 Exercise 4 — Transform a city with always_xy=True

**Task:** Transform the WGS84 coordinate pair for Utrecht into UTM 31N and print the result.

**Steps:**
1. Build a transformer from EPSG:4326 to EPSG:32631 with `always_xy=True`.
2. Pass longitude first and latitude second.

```python
# Hint
transformer = Transformer.from_crs(4326, 32631, always_xy=True)
```

In [ ]:
# 🎯 Exercise 4 — your code here ────────────────────────────────────────────
# 1. Create the transformer.
# 2. Transform Utrecht and print x/y.

In [ ]:
# ✅ Exercise 4 — Solution ────────────────────────────────────────────────────
utrecht = next(row for row in dutch_cities if row['city'] == 'Utrecht')
exercise_transformer = Transformer.from_crs(4326, 32631, always_xy=True)
utrecht_x, utrecht_y = exercise_transformer.transform(utrecht['lon'], utrecht['lat'])
print('Utrecht UTM31N:', round(utrecht_x, 2), round(utrecht_y, 2))

---
## 📖 Section 5 — Distance Comparison

Different distance methods answer slightly different questions because they assume different geometry models.

### 5.1 Haversine, geodesic, and projected Euclidean distance
Haversine approximates Earth as a sphere, `Geod.inv` uses the ellipsoid, and Euclidean distance assumes your projected CRS is appropriate for the study area.

In [ ]:
# 💻 5.1  Compare Haversine and Geod.inv
# ─────────────────────────────────────────────────────────────────────────────
def haversine_km(lon1, lat1, lon2, lat2):
    radius_km = 6371.0088
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2) ** 2
    return 2 * radius_km * math.asin(math.sqrt(a))

geod = Geod(ellps='WGS84')
rotterdam = dutch_cities[1]

haversine_distance_km = haversine_km(ams['lon'], ams['lat'], rotterdam['lon'], rotterdam['lat'])
_, _, geodesic_distance_m = geod.inv(ams['lon'], ams['lat'], rotterdam['lon'], rotterdam['lat'])
print('Haversine km :', round(haversine_distance_km, 3))
print('Geodesic km  :', round(geodesic_distance_m / 1000, 3))

In [ ]:
# 💻 5.2  Compute Euclidean distance in UTM 31N
# ─────────────────────────────────────────────────────────────────────────────
city_lookup = {row['city']: row for row in transformed_cities}
ams_utm = city_lookup['Amsterdam']
rot_utm = city_lookup['Rotterdam']
utm_distance_m = math.dist((ams_utm['utm31n_x'], ams_utm['utm31n_y']), (rot_utm['utm31n_x'], rot_utm['utm31n_y']))

print('UTM Euclidean km:', round(utm_distance_m / 1000, 3))
print('Why the numbers differ: different Earth models and projection assumptions.')

### 🎯 Exercise 5 — Measure another city pair

**Task:** Compute the geodesic and UTM Euclidean distance between Utrecht and Eindhoven.

**Steps:**
1. Use `Geod.inv` on the WGS84 coordinates.
2. Use `math.dist` on the UTM coordinates from `transformed_cities`.

```python
# Hint
_, _, d_m = geod.inv(lon1, lat1, lon2, lat2)
```

In [ ]:
# 🎯 Exercise 5 — your code here ────────────────────────────────────────────
# 1. Look up Utrecht and Eindhoven.
# 2. Compute geodesic and projected distances.

In [ ]:
# ✅ Exercise 5 — Solution ────────────────────────────────────────────────────
utrecht = city_lookup['Utrecht']
eindhoven = city_lookup['Eindhoven']
_, _, geodesic_m = geod.inv(utrecht['lon'], utrecht['lat'], eindhoven['lon'], eindhoven['lat'])
projected_m = math.dist((utrecht['utm31n_x'], utrecht['utm31n_y']), (eindhoven['utm31n_x'], eindhoven['utm31n_y']))

print('Geodesic km :', round(geodesic_m / 1000, 3))
print('UTM km      :', round(projected_m / 1000, 3))

---
## 📖 Section 6 — Shapely Geometry Types

Shapely stores geometry objects and exposes methods for length, area, and coordinate access.

### 6.1 Point, LineString, and Polygon construction
Geometry operations are only as meaningful as the CRS of the numbers you feed into them, so we will use projected coordinates for metric properties.

In [ ]:
# 💻 6.1  Create Point, LineString, and Polygon objects
# ─────────────────────────────────────────────────────────────────────────────
ams_point_utm = Point(ams_utm['utm31n_x'], ams_utm['utm31n_y'])
rot_point_utm = Point(rot_utm['utm31n_x'], rot_utm['utm31n_y'])
route_line = LineString([ams_point_utm, rot_point_utm])
city_polygon = Polygon([
    (ams_utm['utm31n_x'] - 20000, ams_utm['utm31n_y'] - 15000),
    (ams_utm['utm31n_x'] + 20000, ams_utm['utm31n_y'] - 15000),
    (ams_utm['utm31n_x'] + 20000, ams_utm['utm31n_y'] + 15000),
    (ams_utm['utm31n_x'] - 20000, ams_utm['utm31n_y'] + 15000),
])

print('Point type    :', ams_point_utm.geom_type)
print('Line type     :', route_line.geom_type)
print('Polygon type  :', city_polygon.geom_type)

In [ ]:
# 💻 6.2  Inspect geometry coordinates and measurements
# ─────────────────────────────────────────────────────────────────────────────
print('Point coords   :', list(ams_point_utm.coords))
print('Line length km :', round(route_line.length / 1000, 3))
print('Polygon area km²:', round(city_polygon.area / 1_000_000, 3))
print('Polygon bounds :', city_polygon.bounds)

### 🎯 Exercise 6 — Create a metric buffer

**Task:** Create a 50 km buffer around Amsterdam in UTM 31N and print its area in square kilometres.

**Steps:**
1. Use the UTM Point for Amsterdam.
2. Call `.buffer(50_000)` and divide the area by `1_000_000`.

```python
# Hint
buffer_geom = ams_point_utm.buffer(50_000)
```

In [ ]:
# 🎯 Exercise 6 — your code here ────────────────────────────────────────────
# 1. Create the 50 km buffer.
# 2. Print the area in km².

In [ ]:
# ✅ Exercise 6 — Solution ────────────────────────────────────────────────────
ams_buffer_50km = ams_point_utm.buffer(50_000)
print('Buffered geometry type:', ams_buffer_50km.geom_type)
print('Area km²             :', round(ams_buffer_50km.area / 1_000_000, 2))

---
## 📖 Section 7 — Topology and Spatial Predicates

Topology asks how geometries relate: overlap, containment, touching boundaries, and shared interiors.

### 7.1 contains, within, intersects, touches, and DE-9IM
Predicate methods convert geometry relationships into boolean answers you can use in filters, joins, and QA checks.

In [ ]:
# 💻 7.1  Check predicate results between a buffer and city points
# ─────────────────────────────────────────────────────────────────────────────
other_city_points = {
    row['city']: Point(row['utm31n_x'], row['utm31n_y'])
    for row in transformed_cities
}

for city_name, geom in other_city_points.items():
    if city_name == 'Amsterdam':
        continue
    print(
        city_name,
        'within buffer =', geom.within(ams_buffer_50km),
        '| intersects =', geom.intersects(ams_buffer_50km),
    )

In [ ]:
# 💻 7.2  Use touches and DE-9IM relate strings
# ─────────────────────────────────────────────────────────────────────────────
buffer_boundary = ams_buffer_50km.boundary
rotterdam_tiny_buffer = rot_point_utm.buffer(1000)

print('Boundary touches Amsterdam point:', buffer_boundary.touches(ams_point_utm))
print('Amsterdam buffer relates Rotterdam point:', ams_buffer_50km.relate(rot_point_utm))
print('Amsterdam buffer intersects Rotterdam 1 km buffer:', ams_buffer_50km.intersects(rotterdam_tiny_buffer))

### 🎯 Exercise 7 — Test two predicate pairs

**Task:** Evaluate two predicate checks: whether Rotterdam is within Amsterdam's 50 km buffer, and whether the buffer contains the Amsterdam point.

**Steps:**
1. Use the `within()` method on the Rotterdam point.
2. Use `contains()` on the Amsterdam buffer.

```python
# Hint
ams_buffer_50km.contains(ams_point_utm)
```

In [ ]:
# 🎯 Exercise 7 — your code here ────────────────────────────────────────────
# 1. Test Rotterdam against the buffer.
# 2. Test Amsterdam against the same buffer.

In [ ]:
# ✅ Exercise 7 — Solution ────────────────────────────────────────────────────
print('Rotterdam within Amsterdam buffer:', rot_point_utm.within(ams_buffer_50km))
print('Amsterdam buffer contains Amsterdam point:', ams_buffer_50km.contains(ams_point_utm))

---
## 📖 Section 8 — Reusable Module

A local helper module makes CRS-aware validation and transformation reusable in notebooks, scripts, and tests.

### 8.1 Writing `crs_utils.py`
The goal is a small toolkit: validate source records, transform them with a `Transformer`, and compute an XY bounding box.

In [ ]:
# 💻 8.1  Write the crs_utils.py module
# ─────────────────────────────────────────────────────────────────────────────
crs_utils_path = MODULE_DIR / 'crs_utils.py'
crs_utils_source = """
from pyproj import Transformer


class CRSValidationError(Exception):
    pass


def validate_wgs84_record(record):
    lon = float(record['lon'])
    lat = float(record['lat'])
    if not (-180 <= lon <= 180):
        raise CRSValidationError(f'Longitude out of range: {lon}')
    if not (-90 <= lat <= 90):
        raise CRSValidationError(f'Latitude out of range: {lat}')
    normalized = dict(record)
    normalized['lon'] = lon
    normalized['lat'] = lat
    return normalized


def transform_record(record, transformer, x_key='x', y_key='y'):
    normalized = validate_wgs84_record(record)
    x, y = transformer.transform(normalized['lon'], normalized['lat'])
    result = dict(normalized)
    result[x_key] = x
    result[y_key] = y
    return result


def bbox_xy(records, x_key='x', y_key='y'):
    xs = [row[x_key] for row in records]
    ys = [row[y_key] for row in records]
    return min(xs), min(ys), max(xs), max(ys)
"""
crs_utils_path.write_text(crs_utils_source.strip() + '\n', encoding='utf-8')
print('Module written to:', crs_utils_path)

In [ ]:
# 💻 8.2  Import and use crs_utils helpers
# ─────────────────────────────────────────────────────────────────────────────
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import crs_utils
crs_utils = reload(crs_utils)

validated_ams = crs_utils.validate_wgs84_record(dutch_cities[0])
transformed_ams = crs_utils.transform_record(validated_ams, transformer_utm, x_key='utm_x', y_key='utm_y')
print(validated_ams)
print(transformed_ams)

### 🎯 Exercise 8 — Reuse crs_utils.py

**Task:** Use `crs_utils.transform_record()` to transform Groningen to RD New, then compute the bounding box of Amsterdam and Groningen in RD New coordinates.

**Steps:**
1. Transform Groningen with `transformer_rd` and custom key names.
2. Pass two transformed records into `bbox_xy()`.

```python
# Hint
crs_utils.bbox_xy(records, x_key="rd_x", y_key="rd_y")
```

In [ ]:
# 🎯 Exercise 8 — your code here ────────────────────────────────────────────
# 1. Transform Groningen into RD New.
# 2. Compute a small bounding box.

In [ ]:
# ✅ Exercise 8 — Solution ────────────────────────────────────────────────────
groningen = next(row for row in dutch_cities if row['city'] == 'Groningen')
ams_rd = crs_utils.transform_record(dutch_cities[0], transformer_rd, x_key='rd_x', y_key='rd_y')
groningen_rd = crs_utils.transform_record(groningen, transformer_rd, x_key='rd_x', y_key='rd_y')
print('RD bbox:', crs_utils.bbox_xy([ams_rd, groningen_rd], x_key='rd_x', y_key='rd_y'))

---
## 📖 Section 9 — File Formats

Spatial workflows often move the same geometry between text, JSON, and tabular formats.

### 9.1 Exporting and reloading WKT, GeoJSON, and CSV
Portable exports are useful for debugging, quick sharing, and interfacing with non-Python tools like PostGIS or spreadsheets.

In [ ]:
# 💻 9.1  Save geometry data in multiple formats
# ─────────────────────────────────────────────────────────────────────────────
route_wkt_path = OUTPUT_DIR / 'amsterdam_rotterdam_route.wkt'
route_geojson_path = OUTPUT_DIR / 'amsterdam_point.geojson'
route_csv_path = OUTPUT_DIR / 'city_points_projected.csv'

route_wkt_path.write_text(to_wkt(route_line), encoding='utf-8')
route_geojson_path.write_text(
    json.dumps({'type': 'Feature', 'properties': {'city': 'Amsterdam'}, 'geometry': mapping(ams_point_utm)}, indent=2),
    encoding='utf-8',
)
with route_csv_path.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['city', 'lon', 'lat', 'utm31n_x', 'utm31n_y'])
    writer.writeheader()
    writer.writerows([{k: row[k] for k in ['city', 'lon', 'lat', 'utm31n_x', 'utm31n_y']} for row in transformed_cities])

print('Saved WKT, GeoJSON, and CSV exports.')

In [ ]:
# 💻 9.2  Reload geometry from WKT and GeoJSON
# ─────────────────────────────────────────────────────────────────────────────
loaded_line = from_wkt(route_wkt_path.read_text(encoding='utf-8'))
loaded_feature = json.loads(route_geojson_path.read_text(encoding='utf-8'))
loaded_point = shape(loaded_feature['geometry'])

print('Loaded line type :', loaded_line.geom_type)
print('Loaded point type:', loaded_point.geom_type)
print('Loaded point XY  :', list(loaded_point.coords)[0])

### 🎯 Exercise 9 — Round-trip another geometry

**Task:** Export the Rotterdam point to GeoJSON and then reload it with `shape()`.

**Steps:**
1. Create a GeoJSON feature dict using `mapping(rot_point_utm)`.
2. Write it to disk, read it back, and convert the geometry with `shape()`.

```python
# Hint
feature = {"type": "Feature", "geometry": mapping(rot_point_utm), ...}
```

In [ ]:
# 🎯 Exercise 9 — your code here ────────────────────────────────────────────
# 1. Write a Rotterdam GeoJSON file.
# 2. Reload it and inspect the geometry type.

In [ ]:
# ✅ Exercise 9 — Solution ────────────────────────────────────────────────────
rotterdam_geojson_path = OUTPUT_DIR / 'rotterdam_point.geojson'
rotterdam_feature = {
    'type': 'Feature',
    'properties': {'city': 'Rotterdam'},
    'geometry': mapping(rot_point_utm),
}
rotterdam_geojson_path.write_text(json.dumps(rotterdam_feature, indent=2), encoding='utf-8')
reloaded_rotterdam = shape(json.loads(rotterdam_geojson_path.read_text(encoding='utf-8'))['geometry'])
print('Reloaded geometry:', reloaded_rotterdam.geom_type, list(reloaded_rotterdam.coords)[0])

---
## 📖 Section 10 — Logging and Testing

Spatial code should be observable and verifiable, especially when transformations feed later analysis or database loads.

### 10.1 Log transform steps and test CRS helpers
We log what happened to each record and then prove the helper module behaves correctly with automated tests.

In [ ]:
# 💻 10.1  Configure a Week 4 logger and log a transform pipeline
# ─────────────────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'week4.log'
logger = logging.getLogger('gpm.week4')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

formatter = logging.Formatter('%(levelname)s | %(asctime)s | %(message)s')
file_handler = logging.FileHandler(log_path, encoding='utf-8')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

for row in dutch_cities:
    transformed = crs_utils.transform_record(row, transformer_utm, x_key='utm_x', y_key='utm_y')
    logger.info('Transformed %s to UTM31N: (%.2f, %.2f)', transformed['city'], transformed['utm_x'], transformed['utm_y'])

print('Log file:', log_path)

In [ ]:
# 💻 10.2  Write and run unit tests for crs_utils.py
# ─────────────────────────────────────────────────────────────────────────────
test_file_path = TEST_DIR / 'test_crs_utils.py'
test_file_source = """
import sys
import unittest
from pathlib import Path
from pyproj import Transformer

COURSE_DIR = Path(__file__).resolve().parents[1]
MODULE_DIR = COURSE_DIR / 'local_modules'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import crs_utils


class TestCRSUtils(unittest.TestCase):
    def setUp(self):
        self.record = {'city': 'Amsterdam', 'lon': 4.9041, 'lat': 52.3676}
        self.bad_record = {'city': 'Broken', 'lon': 500.0, 'lat': 52.0}
        self.transformer = Transformer.from_crs(4326, 32631, always_xy=True)

    def test_validate_wgs84_record_normalizes_floats(self):
        result = crs_utils.validate_wgs84_record(self.record)
        self.assertIsInstance(result['lon'], float)
        self.assertEqual(result['city'], 'Amsterdam')

    def test_validate_wgs84_record_rejects_invalid_lon(self):
        with self.assertRaises(crs_utils.CRSValidationError):
            crs_utils.validate_wgs84_record(self.bad_record)

    def test_transform_record_adds_projected_keys(self):
        result = crs_utils.transform_record(self.record, self.transformer, x_key='utm_x', y_key='utm_y')
        self.assertIn('utm_x', result)
        self.assertIn('utm_y', result)


if __name__ == '__main__':
    unittest.main()
"""
test_file_path.write_text(test_file_source.strip() + '\n', encoding='utf-8')

suite = unittest.defaultTestLoader.discover(str(TEST_DIR), pattern='test_crs_utils.py')
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### 🎯 Exercise 10 — Add one assertRaises check

**Task:** Write an inline `unittest.TestCase` that confirms `validate_wgs84_record()` raises `CRSValidationError` when latitude is outside -90..90.

**Steps:**
1. Create a small `unittest.TestCase` subclass.
2. Use `self.assertRaises(...)` in one test method.

```python
# Hint
with self.assertRaises(crs_utils.CRSValidationError):
    crs_utils.validate_wgs84_record(...)
```

In [ ]:
# 🎯 Exercise 10 — your code here ────────────────────────────────────────────
# 1. Define the inline test class.
# 2. Run the test suite.

In [ ]:
# ✅ Exercise 10 — Solution ────────────────────────────────────────────────────
class InlineCRSTests(unittest.TestCase):
    def test_invalid_latitude_raises(self):
        with self.assertRaises(crs_utils.CRSValidationError):
            crs_utils.validate_wgs84_record({'city': 'Broken', 'lon': 4.9, 'lat': 123.0})

unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(InlineCRSTests)
)

---
## 🔬 Mini-Lab — Netherlands Cities CRS Comparison

**Scenario:** You are preparing a briefing for a Dutch planning team. They need to understand how the same five city locations behave in WGS84, UTM 31N, and RD New before they commit to a national analysis workflow.

**Tasks:**
1. Create a list of 5 Dutch cities with WGS84 coordinates
2. Validate all coordinates with `crs_utils`
3. Transform to UTM 31N (`EPSG:32631`) and to RD New (`EPSG:28992`)
4. Compute Amsterdam→Rotterdam distance with `Geod.inv` AND UTM Euclidean
5. Create a 50 km Shapely buffer around Amsterdam in UTM 31N
6. Check which other cities fall within the buffer
7. Export results to a GeoJSON file

In [ ]:
# 🔬 Step 1 — Create five Dutch city records
# ─────────────────────────────────────────────────────────────────────────────
lab_cities = [dict(row) for row in dutch_cities]
pprint(lab_cities)

In [ ]:
# 🔬 Step 2 — Validate all coordinates with crs_utils
# ─────────────────────────────────────────────────────────────────────────────
validated_lab_cities = [crs_utils.validate_wgs84_record(row) for row in lab_cities]
print('Validated records:', len(validated_lab_cities))

In [ ]:
# 🔬 Step 3 — Transform to UTM 31N and RD New
# ─────────────────────────────────────────────────────────────────────────────
lab_transformed = []
for row in validated_lab_cities:
    utm_row = crs_utils.transform_record(row, transformer_utm, x_key='utm_x', y_key='utm_y')
    rd_row = crs_utils.transform_record(row, transformer_rd, x_key='rd_x', y_key='rd_y')
    lab_transformed.append({**utm_row, 'rd_x': rd_row['rd_x'], 'rd_y': rd_row['rd_y']})

pprint(lab_transformed)

In [ ]:
# 🔬 Step 4 — Compute Amsterdam→Rotterdam distances
# ─────────────────────────────────────────────────────────────────────────────
lab_lookup = {row['city']: row for row in lab_transformed}
ams_lab = lab_lookup['Amsterdam']
rot_lab = lab_lookup['Rotterdam']
_, _, geodesic_m = geod.inv(ams_lab['lon'], ams_lab['lat'], rot_lab['lon'], rot_lab['lat'])
projected_m = math.dist((ams_lab['utm_x'], ams_lab['utm_y']), (rot_lab['utm_x'], rot_lab['utm_y']))

print('Geod.inv km :', round(geodesic_m / 1000, 3))
print('UTM km      :', round(projected_m / 1000, 3))

In [ ]:
# 🔬 Step 5 — Create a 50 km Amsterdam buffer in UTM 31N
# ─────────────────────────────────────────────────────────────────────────────
ams_lab_point = Point(ams_lab['utm_x'], ams_lab['utm_y'])
ams_lab_buffer = ams_lab_point.buffer(50_000)
print('Buffer area km²:', round(ams_lab_buffer.area / 1_000_000, 2))

In [ ]:
# 🔬 Step 6 — Check which cities fall within the buffer
# ─────────────────────────────────────────────────────────────────────────────
within_results = []
for row in lab_transformed:
    if row['city'] == 'Amsterdam':
        continue
    point = Point(row['utm_x'], row['utm_y'])
    within_results.append({'city': row['city'], 'within_50km': point.within(ams_lab_buffer)})

pprint(within_results)

In [ ]:
# 🔬 Step 7 — Export comparison results to GeoJSON
# ─────────────────────────────────────────────────────────────────────────────
lab_geojson_features = []
for row in lab_transformed:
    point = Point(row['utm_x'], row['utm_y'])
    feature = {
        'type': 'Feature',
        'properties': {
            'city': row['city'],
            'lon': row['lon'],
            'lat': row['lat'],
            'utm_x': row['utm_x'],
            'utm_y': row['utm_y'],
            'rd_x': row['rd_x'],
            'rd_y': row['rd_y'],
        },
        'geometry': mapping(point),
    }
    lab_geojson_features.append(feature)

lab_geojson_path = OUTPUT_DIR / 'lab_netherlands_cities_utm31n.geojson'
lab_geojson_path.write_text(
    json.dumps({'type': 'FeatureCollection', 'features': lab_geojson_features}, indent=2),
    encoding='utf-8',
)
print('Exported lab GeoJSON:', lab_geojson_path)

### 🚀 Extension Ideas

- Compare UTM 31N with Web Mercator (EPSG:3857) to see distortion more clearly
- Add an RD New polygon study area and test `intersects()` with city buffers
- Export the same cities to a PostGIS-ready CSV with `srid` metadata

---
## ✅ Week 4 Summary

| Topic | Key concepts mastered |
|-------|-----------------------|
| WGS84 datum | Longitude, latitude, and ellipsoidal thinking for global storage |
| geographic CRS | Angular coordinates, axis metadata, and degree units |
| projected CRS | Planar x/y coordinates in metres for local measurement |
| PyProj CRS | `CRS.from_epsg()`, axis info, units, WKT export |
| Transformer | `Transformer.from_crs(..., always_xy=True)` for safe reprojection |
| Geod.inv | Ellipsoidal distance measurement in metres and kilometres |
| Shapely geometry | Point, LineString, Polygon creation plus length and area |
| topology predicates | `contains`, `within`, `intersects`, `touches`, `relate` |
| crs_utils module | Reusable validation, transform, and bounding-box helpers |
| file export | Portable WKT, GeoJSON, and CSV round-tripping |

---

### ☑️ Self-assessment checklist

- [ ] Explain the difference between geographic and projected CRS without notes
- [ ] Create a `pyproj.CRS` from an EPSG code and read its name and unit
- [ ] Transform a WGS84 lon/lat pair to UTM 31N using `always_xy=True`
- [ ] Compute a geodesic distance with `pyproj.Geod.inv` in kilometres
- [ ] Create a Shapely Point, buffer it by 50 km in UTM, and read the area
- [ ] Test `polygon.contains(point)` for two different pairs
- [ ] Import and use a function from your local `crs_utils.py`
- [ ] Save a Shapely geometry to GeoJSON and reload it with `shape()`

---

## 📚 Week 5 Preview

| Topic | What you will learn |
|-------|---------------------|
| GeoPandas GeoDataFrame | Create spatial tables with geometry columns and CRS metadata |
| read_file / to_file | Load and export vector data with GeoPandas and Fiona |
| Shapely advanced ops | Buffer, overlay, dissolve, and union workflows |
| spatial joins | Match features by location instead of just attribute keys |
| Rasterio basics | Open raster grids and inspect bands, transforms, and metadata |

---

## 📖 Further Reading

| Resource | Why |
|----------|-----|
| [PyProj docs](https://pyproj4.github.io/pyproj/stable/) | CRS metadata, transformers, and geodesic tools |
| [Shapely docs](https://shapely.readthedocs.io/) | Geometry creation, predicates, and measurements |
| [EPSG registry](https://epsg.io/) | Look up CRS definitions and metadata by code |
| [Understanding map projections](https://www.axismaps.com/guide/map-projections) | Intuition for distortion and projection choices |
| [GeoJSON spec](https://geojson.org/) | Geometry and feature structure for interchange |
| [Real Python coordinate transforms](https://realpython.com/python-geocoding/) | Broader context for coordinates and geocoding workflows |

---

*Geospatial Python Mastery — Week 4 of 10*
*For educational use. Please keep feedback to help improve future iterations.*